In [ ]:
# =========================================================
# Import all dependencies
# =========================================================
import os
import numpy as np
from tqdm import tqdm
import tensorflow as tf
import matplotlib as mpl
import matplotlib.pyplot as plt
from cdshealpix import healpix_to_skycoord
from astra.src.preprocessing import augmentation
from astra.utils.helper import deserialize_for_inference

In [ ]:
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['STIXGeneral', 'DejaVu Serif', 'serif']
plt.rcParams['mathtext.fontset'] = 'stix'

## Load TfRecords 

In [ ]:
path_to_save = "/media3/majumder/astra/figures/"
path_to_read = "/media3/majumder/dataset/resampled_data_12K/val/AGN/"

In [ ]:
glob_pattern = os.path.join(path_to_read, 'chunk_*.record') 
filenames = tf.data.Dataset.list_files(glob_pattern, shuffle=False)
if len(filenames) > 1:
    raw_dataset = tf.data.TFRecordDataset(filenames[0])
else:
    raw_dataset = tf.data.TFRecordDataset(filenames)
parsed_dataset = raw_dataset.map(deserialize_for_inference)
extracted_dict = {}
for data_dict in parsed_dataset.skip(1).take(1):
    extracted_dict['input_id'] = data_dict['input_id'].numpy() 
    extracted_dict['id'] = data_dict['id'].numpy()
    extracted_dict['label'] = data_dict['label'].numpy().decode('utf-8')
    extracted_dict['last_index'] = data_dict['last_index'].numpy()
    extracted_dict['bands'] = [band.decode('utf-8') for band in data_dict['bands'].numpy()]
    

print("Keys in extracted dictionary:", extracted_dict.keys())
print("Shape of input_id (Light Curve Data):", extracted_dict['input_id'].shape)
print("Available Bands:", extracted_dict['bands'])
print("Last Index:", extracted_dict['last_index'])
print("ID:", extracted_dict['id'])
print("Label:", extracted_dict['label'])

#### Get skycoord from healpix id and search in the [SNAD ZTF Viewer](https://ztf.snad.space/) to get the PS1 & Gaia ID

In [ ]:
skycoord= healpix_to_skycoord(782376312830009490, depth=29)
print(skycoord)
# ZTF ID: 791112300000247
# 668003412881850176

## Plot the original light curve

In [ ]:
PLOT_CONFIG = {
    'g': {'color': '#7FCB06', 'marker': 'H', 'markersize': 6, 'alpha': 0.8}, # Green
    'r': {'color': '#F77808', 'marker': 'o', 'markersize': 6, 'alpha': 0.9}, # Red
    'i': {'color': '#AF5AE8', 'marker': 'h', 'markersize': 6, 'alpha': 0.8}, # Purple
    'errorbar_kwargs': {
        'fmt': 'none',     # Don't draw connecting lines
        'elinewidth': 0.6, # Error bar line width
        'capsize': 0,      # Cap size on error bars (0 is modern/clean)
        'zorder': 1        # Keep error bars behind markers
    },
    'labels': {
        'fontsize': 16,
        'title_size': 16,
        'legend_size': 14,
        'tick_size': 12
    }
}
MJD_OFFSET = 58_000
# ==========================================
# 2. PLOTTING FUNCTION
# ==========================================
def plot_original_lightcurve(extracted_dict, save_path=None):
    
    # Extract data from your dictionary
    input_id = extracted_dict['input_id']
    bands = extracted_dict['bands']
    last_indices = extracted_dict['last_index']
    
    # Create Figure
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    
    start_idx = 0
    
    # Loop through available bands and slice the array using last_indices
    for i, band in enumerate(bands):
        # Calculate end index (inclusive of the value at last_index, so +1)
        end_idx = last_indices[i] + 1
        
        # Slice the dataset
        band_data = input_id[start_idx:end_idx]
        
        # Extract columns: mjd, mag, magerr
        mjd = band_data[:, 0] + MJD_OFFSET 
        if band == 'i':
            mag = band_data[:, 1]  # Shift i-band magnitudes for better visibility
            legend_label = f'$(i - 1)$'
        elif band == 'g':
            mag = band_data[:, 1]  # Shift r-band magnitudes slightly
            legend_label = f'$(g + 1)$'
        else:
            mag = band_data[:, 1] 
            legend_label = f'$r$'
        
        magerr = band_data[:, 2]/10_000 
        print(f"\nBand: {band}, MJD range: {mjd.min()} - {mjd.max()}, Mag range: {mag.min()} - {mag.max()}")
        
        style = PLOT_CONFIG.get(band, {'color': 'black', 'marker': 'o', 'markersize': 4, 'alpha': 1.0})
        
        ax.errorbar(
            mjd, mag, yerr=magerr,
            ecolor=style['color'],
            alpha=style['alpha'],
            **PLOT_CONFIG['errorbar_kwargs']
        )
        
        # Plot Scatter markers over the error bars
        ax.scatter(
            mjd, mag,
            color=style['color'],
            marker=style['marker'],
            s=style['markersize']**2, # Scatter takes area (size^2)
            alpha=style['alpha'],
            label=legend_label, # LaTeX formatted legend
            zorder=2,
            edgecolors='white', # Adds a tiny border to markers for clarity
            linewidths=0.2
        )
        
        
        start_idx = end_idx
    
    ax.invert_yaxis()
    # ax.set_xlim(left=58_150, right=60_000)
    # ax.set_ylim(21.3, 17.0)
    
    # Labels and Title
    ax.set_xlabel('Modified Julian Date $(t)$', fontsize=PLOT_CONFIG['labels']['fontsize'], labelpad=10)
    ax.set_ylabel(f"Apparent Magnitude $(m)$", fontsize=PLOT_CONFIG['labels']['fontsize'], labelpad=10)
    # ax.set_title('ZTF OID: 791112300000247', fontsize=PLOT_CONFIG['labels']['title_size'], pad=15)
    
    # Tick formatting
    ax.tick_params(axis='both', which='major', labelsize=PLOT_CONFIG['labels']['tick_size'], length=6)
    ax.tick_params(axis='both', which='minor', length=3)
    ax.legend(
                loc='lower center', 
                bbox_to_anchor=(0.5, 1.0), # Anchors legend just above the top plot
                ncol=3,                     # Puts legend items in a single horizontal row
                fontsize=PLOT_CONFIG['labels']['legend_size'], 
                # frameon=True, 
                edgecolor='gray', 
                borderpad=0.25, 
                handletextpad=0.25,
                # shadow=True,
                fancybox=False
            )
    # Legend
    # ax.legend(
    #     loc='best', 
    #     fontsize=PLOT_CONFIG['labels']['legend_size'],
    #     frameon=True, 
    #     edgecolor='black',
    #     fancybox=False # Square box for formal publication look
    # )
    # ax.legend(fontsize=PLOT_CONFIG['labels']['legend_size'], loc='best', edgecolor='black', 
    #       labelspacing=0.25, shadow=True, borderpad=0.25, handletextpad=0.25)
    
    plt.tight_layout()
    
    # Save or Show
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"Plot saved successfully to {save_path}")
    
    plt.show()
    plt.close()



In [ ]:
output_filename = os.path.join(path_to_save, f'original_lc.pdf')
plot_original_lightcurve(extracted_dict, output_filename)

## Generate the views - Global (GV) and Local (LV) - using data augmentation pipeline

In [ ]:
#
# This script demonstrates how to use the contrastive_data_loader function
#
n_views = 3
batch_size = 5
buffer_size = 1000 

apply_binning=[False, False, True]
apply_outlier=[False, False, True]
apply_white_noise=[True, False, True]

bin_widths = [5, 5, 5]
drop_rates = [0.0, 0.0, 0.50]
noise_levels = [0.05, 0.0, 0.10] 

maxlens=[{'g': 300, 'r': 350, 'i': 50}, {'g': 150, 'r': 175, 'i': 25}, {'g': 300, 'r': 350, 'i': 50}] 
build_seq_len=tf.cast(sum(maxlens[0].values()), tf.int32)  

In [ ]:
glob_pattern = os.path.join(path_to_read, 'chunk_*.record') 
filenames = tf.data.Dataset.list_files(glob_pattern, shuffle=False)
if len(filenames) > 1:
    raw_dataset = tf.data.TFRecordDataset(filenames[0])
else:
    raw_dataset = tf.data.TFRecordDataset(filenames)

loaders = []
for i in range(n_views):
    aug_fn = lambda data, idx=i: augmentation(data,
                                                apply_white_noise=apply_white_noise[idx],
                                                noise_level=noise_levels[idx],
                                                apply_binning=apply_binning[idx],
                                                apply_outlier=apply_outlier[idx],
                                                maxlen=maxlens[idx],
                                                bin_width=bin_widths[idx],
                                                drop_data=drop_rates[idx],
                                                for_inference=False)
    view_loader = raw_dataset.map(aug_fn)
    loaders.append(view_loader)
# --------------------------------------------- Zipping and Batching ------------------------------------------
extracted_augmented_views=[]
for i, loader in enumerate(loaders):
    for sample in loader.skip(1).take(1):
        
        view_dict = {
            'times': sample['times'].numpy().flatten(),           # MJD
            'input': sample['input'].numpy().flatten(),           # Standardized Magnitude
            'band_info': sample['band_info'].numpy().flatten(),   # Band indicators
            'mask': sample['mask'].numpy().flatten()              # Used to ignore padding
    
        }
    
        extracted_augmented_views.append(view_dict)

print(f"Successfully extracted {len(extracted_augmented_views)} augmented views!")

## Plot the views

In [ ]:
# Configuration for Augmented Plot
AUG_PLOT_CONFIG = {
    'color': '#1f77b4',       # A professional, publication-safe blue
    'marker': 'o',            # Circle marker
    'markersize': 6,        # Slightly smaller since there might be many points
    'alpha': 0.8,             # Transparency to show density/overlapping points
    'labels': {
        'fontsize': 16,
        'title_size': 18,
        'tick_size': 12,
        'view_text_size': 14
    }
}

# ==========================================
# 2. PLOTTING FUNCTION
# ==========================================
def plot_augmented_views(extracted_augmented_views, save_path=None):
    """
    Plots 3 augmented views of a light curve in a 3x1 subplot layout.
    Filters out masked points (mask == 1).
    """
    
    # Create Figure with 3 rows, 1 column. 
    # figsize=(7, 7) fits well across a 2-column paper span (\begin{figure*})
    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(8, 6), dpi=300, sharex=True)
    
    # Loop through the 3 extracted views and the 3 subplots
    for i, view in enumerate(extracted_augmented_views):
        if i == 0:
            aug_text = f"Global View (GV)"
            box_size = (0.770, 0.07)
        elif i == 1:
            aug_text = f"Local View 1 (LV1)"
            box_size = (0.750, 0.07)
        else:
            aug_text = f"Local View 2 (LV2)"
            box_size = (0.750, 0.07)
        ax = axes[i]
        
        times = view['times']
        inputs = view['input']
        mask = view['mask']
        band_info = view['band_info']
        
        # Map the unique float values to their respective bands
        band_mapping = {
            'g': 3.6763716,
            'r': 3.8038926,
            'i': 3.893708
        }
        
        # Loop through the bands to plot them individually with their colors
        for band, target_val in band_mapping.items():
            
            # FILTER LOGIC: Keep only unmasked segments AND matching band_info
            # (using np.isclose with a small tolerance for floating point safety)
            valid_idx = (mask == 0) & np.isclose(band_info, target_val, atol=1e-4)
            
            plot_times = times[valid_idx] + MJD_OFFSET  # Shift MJD for better readability
            plot_inp = inputs[valid_idx] 

            if band == 'i':
                plot_inputs = plot_inp - 1 # Shift i-band magnitudes for better visibility
                legend_label = f'$(i - 1)$'
            elif band == 'g':
                plot_inputs = plot_inp + 1 # Shift r-band magnitudes slightly
                legend_label = f'$(g + 1)$'
            else:
                plot_inputs = plot_inp 
                legend_label = f'$r$'
            
            # Skip if a specific band was entirely dropped during augmentation
            if len(plot_times) == 0:
                continue
                
            # Get styles from your original PLOT_CONFIG
            style = PLOT_CONFIG.get(band)
            
            # Plot Scatter
            ax.scatter(
                plot_times, plot_inputs,
                color=style['color'],
                marker=style['marker'],
                s=style['markersize']**2.2, 
                alpha=style['alpha'],
                edgecolors='white', 
                linewidths=0.2,
                label=legend_label if i == 0 else "" # Legend labels only on the first subplot
            )
            
        # Add a legend only to the top subplot
        if i == 0:
            ax.legend(loc='upper right', fontsize=AUG_PLOT_CONFIG['labels']['tick_size'], 
                      frameon=True, edgecolor='black', fancybox=False)
        
        # INVERT Y-AXIS (Standard for scaled/unscaled magnitudes)
        ax.invert_yaxis()
        ax.set_xlim(left=58_150, right=60_000)
        ax.set_ylim(2.2, -2.2)
        # ax.set_ylim(21.3, 17.0)
        
        # Subplot Formatting
        # ax.set_ylabel(f"Scaled Apparent Mag. $(m')$", fontsize=AUG_PLOT_CONFIG['labels']['fontsize'], labelpad=10)
        # Set Y-axis label ONLY on the middle subplot (index 1)
        if i == 1:
            ax.set_ylabel(f"Centered Apparent Magnitude $(m')$", fontsize=AUG_PLOT_CONFIG['labels']['fontsize'], labelpad=10)
        ax.tick_params(axis='both', which='major', labelsize=AUG_PLOT_CONFIG['labels']['tick_size'], length=6)
        ax.tick_params(axis='both', which='minor', length=3)
        
        # Add a text box inside the plot to label the view (saves space vs external titles)
        ax.text(
            box_size[0], box_size[1], aug_text, 
            transform=ax.transAxes, # Use relative axis coordinates
            fontsize=AUG_PLOT_CONFIG['labels']['view_text_size'],
            verticalalignment='bottom',
            bbox=dict(boxstyle='square,pad=0.2', facecolor='white', alpha=0.7, edgecolor='gray')
        )
        if i == 0:
            ax.legend(
                loc='lower center', 
                bbox_to_anchor=(0.5, 1.0), # Anchors legend just above the top plot
                ncol=3,                     # Puts legend items in a single horizontal row
                fontsize=AUG_PLOT_CONFIG['labels']['view_text_size'], 
                # frameon=True, 
                edgecolor='gray', 
                borderpad=0.25, 
                handletextpad=0.25,
                fancybox=False
            )

    # Set X-axis label only on the bottom subplot
    axes[-1].set_xlabel('Modified Julian Date $(t)$', fontsize=AUG_PLOT_CONFIG['labels']['fontsize'], labelpad=10)
    
    # Set a global title
    # fig.suptitle('Augmented Light Curve Variations', fontsize=AUG_PLOT_CONFIG['labels']['title_size'], y=0.93)
    
    # Minimize vertical space between subplots for a highly cohesive, publication-ready look
    plt.subplots_adjust(hspace=0.06)
    
    # Save or Show
    if save_path:
        # plt.savefig(save_path, bbox_inches='tight')
        plt.tight_layout()
        plt.savefig(output_filename, dpi=300, transparent=True, bbox_inches='tight')
        print(f"Plot saved successfully to {save_path}")
    
    plt.show()
    plt.close()

In [ ]:
output_filename = os.path.join(path_to_save, f'aug_lc.png')
plot_augmented_views(extracted_augmented_views, output_filename)